In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [2]:
# Load data
clients = pd.read_csv("../data/client_data (1).csv")
price_data = pd.read_csv("../data/price_data (1) (1).csv")

print("Clients shape:", clients.shape)
print("Price data shape:", price_data.shape)

Clients shape: (14606, 26)
Price data shape: (193002, 8)


# 1. Missing Values Analysis

In [3]:
# Check missing values
print("=" * 60)
print("MISSING VALUES IN CLIENTS DATA")
print("=" * 60)
missing_clients = clients.isnull().sum()
missing_pct = (missing_clients / len(clients)) * 100
missing_df = pd.DataFrame({
    'Column': missing_clients.index,
    'Missing_Count': missing_clients.values,
    'Missing_%': missing_pct.values
}).sort_values('Missing_%', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])
print("\nNo missing values!" if missing_clients.sum() == 0 else "\nMissing values detected!")

print("\n" + "=" * 60)
print("MISSING VALUES IN PRICE DATA")
print("=" * 60)
missing_price = price_data.isnull().sum()
print(missing_price[missing_price > 0] if missing_price.sum() > 0 else "No missing values!")

MISSING VALUES IN CLIENTS DATA
Empty DataFrame
Columns: [Column, Missing_Count, Missing_%]
Index: []

No missing values!

MISSING VALUES IN PRICE DATA
No missing values!


# 2. Outlier Detection (IQR Method)

In [4]:
# Function to detect outliers using IQR method
def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    return len(outliers), lower_bound, upper_bound, outliers

# Numerical columns
numerical_cols = clients.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [c for c in numerical_cols if c != 'id']  # Exclude ID

print("=" * 80)
print("OUTLIER DETECTION (IQR Method - 1.5 × IQR)")
print("=" * 80)

outlier_summary = []
for col in numerical_cols:
    if col in clients.columns:
        count, lower, upper, _ = detect_outliers_iqr(clients, col)
        pct = (count / len(clients)) * 100
        if count > 0:
            outlier_summary.append({
                'Column': col,
                'Outlier_Count': count,
                'Outlier_%': pct,
                'Lower_Bound': lower,
                'Upper_Bound': upper
            })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_%', ascending=False)
    print(outlier_df.to_string(index=False))
else:
    print("No significant outliers detected!")

OUTLIER DETECTION (IQR Method - 1.5 × IQR)
                        Column  Outlier_Count  Outlier_%   Lower_Bound  Upper_Bound
                   nb_prod_act           3175  21.737642      1.000000     1.000000
                  cons_gas_12m           2612  17.883062      0.000000     0.000000
                      cons_12m           2084  14.268109 -46958.750000 93397.250000
               cons_last_month           2051  14.042174  -5074.500000  8457.500000
                       pow_max           1535  10.509380      2.491250    29.181250
                         churn           1419   9.715186      0.000000     0.000000
            forecast_cons_year           1298   8.886759  -2618.625000  4364.375000
                      imp_cons           1215   8.318499   -290.970000   484.950000
             forecast_cons_12m           1054   7.216213  -2365.197500  5261.982500
                    net_margin           1022   6.997124   -237.865000   531.675000
          margin_gross_pow_ele   

# 3. Data Consistency Checks

In [5]:
# Convert date columns to datetime
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    if col in clients.columns:
        clients[col] = pd.to_datetime(clients[col], errors='coerce')

print("=" * 80)
print("DATE CONSISTENCY CHECKS")
print("=" * 80)

# Check: date_activ should be before date_end
invalid_dates = clients[clients['date_activ'] >= clients['date_end']]
print(f"\n1. Cases where date_activ >= date_end: {len(invalid_dates)}")
if len(invalid_dates) > 0:
    print(invalid_dates[['id', 'date_activ', 'date_end']].head())

# Contract duration
clients['contract_duration_days'] = (clients['date_end'] - clients['date_activ']).dt.days
print(f"\n2. Contract duration statistics:")
print(f"   Min: {clients['contract_duration_days'].min()} days")
print(f"   Max: {clients['contract_duration_days'].max()} days")
print(f"   Mean: {clients['contract_duration_days'].mean():.2f} days")
print(f"   Median: {clients['contract_duration_days'].median():.2f} days")

# Check: negative or zero contract duration
invalid_contract = clients[clients['contract_duration_days'] <= 0]
print(f"\n3. Cases with contract duration <= 0: {len(invalid_contract)}")

DATE CONSISTENCY CHECKS

1. Cases where date_activ >= date_end: 0

2. Contract duration statistics:
   Min: 731 days
   Max: 4795 days
   Mean: 2007.54 days
   Median: 1828.50 days

3. Cases with contract duration <= 0: 0
